# Tutorial: Differential Privacy - Noise and Accounting

**Prerequisites**: Tutorial 01 (Gradient Clipping), Basic understanding of privacy concepts

---

## Overview

In Tutorial 01, we learned **gradient clipping** - how to bound the sensitivity of per-example gradients. But clipping alone doesn't provide privacy! To achieve **differential privacy (DP)**, we need two more components:

1. ✅ **Gradient clipping** (Tutorial 01) - Bounds sensitivity
2. 🎯 **Noise injection** (This tutorial) - Adds calibrated Gaussian noise
3. 📊 **Privacy accounting** (This tutorial) - Tracks privacy budget (ε, δ)

This tutorial focuses on **understanding and using** noise and accounting - the theoretical foundations of DP-SGD. We'll save the complete training loop for Tutorial 03.

**Learning objectives**:
1. Understand why noise is essential for differential privacy
2. Use `gaussian_noise()` to add calibrated noise to gradients
3. Use privacy accountants (`PLDAccountant`, `RDPAccountant`) to track privacy
4. Understand truncated Poisson sampling for stable batch sizes
5. Use calibration functions to find optimal hyperparameters for target privacy budgets

**What you'll build**:
- Manual noise addition to clipped gradients
- Privacy accounting for different sampling strategies
- Automated hyperparameter calibration for privacy budgets

---

## What is Differential Privacy?

**Informal definition**: An algorithm is differentially private if its output is nearly identical whether or not any single individual's data is included.

**Formal definition**: A randomized algorithm $\mathcal{M}$ satisfies $(\varepsilon, \delta)$-differential privacy if for all datasets $D$ and $D'$ differing in one row, and all outputs $S$:

$$P[\mathcal{M}(D) \in S] \leq e^\varepsilon \cdot P[\mathcal{M}(D') \in S] + \delta$$

**Key parameters**:
- **ε (epsilon)**: Privacy loss - smaller is better (typical: 1-10)
- **δ (delta)**: Failure probability - very small (typical: 1e-5 to 1e-7)

**Intuition**:
- ε ≈ 0: Perfect privacy (output independent of any individual)
- ε ≈ 1: Strong privacy
- ε ≈ 10: Weak privacy
- δ: Probability that privacy guarantee fails (should be << 1/dataset_size)

---

## Why Do We Need Noise?

**Gradient clipping alone is NOT private!**

Consider two datasets:
- Dataset A: [example_1, example_2, example_3]
- Dataset B: [example_1, example_2, example_4]  # different third example

If we clip gradients to norm 1.0:
```python
grad_A = clip(grad_1) + clip(grad_2) + clip(grad_3)
grad_B = clip(grad_1) + clip(grad_2) + clip(grad_4)
```

The outputs are **deterministic and different** - an adversary can tell which dataset was used!

**Solution**: Add Gaussian noise scaled to the sensitivity:
```python
grad_A_private = grad_A + N(0, σ²)  # σ = noise_multiplier × clip_norm
grad_B_private = grad_B + N(0, σ²)
```

Now the outputs are **random and similar** - harder to distinguish!

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F

# Opaque imports
import opaque.accounting as acc
from opaque import make_functional, clipped_grad, gaussian_noise
from opaque import bounded_gaussian_noise

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Opaque imported successfully!")

---

## Part 1: Understanding Gaussian Noise

Let's start by understanding how Gaussian noise works.

In [3]:
# Create a simple gradient (simulating clipped and summed gradients)
grad = torch.tensor([1.0, 2.0, 3.0])

print(f"Original gradient: {grad}")
print(f"L2 norm: {grad.norm().item():.4f}")

Original gradient: tensor([1., 2., 3.])
L2 norm: 3.7417


In [ ]:
# Add Gaussian noise with stddev = 1.0
noise_fn, state = gaussian_noise(stddev=1.0)
noisy_grad, state = noise_fn(grad, state)

print(f"Noisy gradient: {noisy_grad}")
print(f"Noise added: {noisy_grad - grad}")
print(f"L2 norm of noise: {(noisy_grad - grad).norm().item():.4f}")

### Visualizing Noise Distribution

Let's see what happens when we add noise many times:

In [ ]:
# Generate many noisy versions
n_samples = 1000
original = torch.tensor([5.0])  # Single value for simplicity
stddev = 1.0

noisy_samples = []
for i in range(n_samples):
  noise_fn, state = gaussian_noise(stddev=stddev, generator=i)
  noisy, state = noise_fn(original, state)
  noisy_samples.append(noisy.item())

noisy_samples = np.array(noisy_samples)

# Plot histogram
plt.figure(figsize=(10, 5))
plt.hist(noisy_samples, bins=50, density=True, alpha=0.7, edgecolor='black')
plt.axvline(original.item(), color='red', linestyle='--', linewidth=2,
            label=f'True value = {original.item()}')
plt.xlabel('Value', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.title(f'Distribution of Noisy Gradient (stddev={stddev})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean of noisy samples: {noisy_samples.mean():.4f} (should be ≈ {original.item()})")
print(f"Std of noisy samples:  {noisy_samples.std():.4f} (should be ≈ {stddev})")

**Key insight**:
- Noise is centered around the true value (unbiased)
- Higher stddev = more noise = better privacy, but worse utility
- This is the **privacy-utility tradeoff**

---

## Part 2: DP-SGD Algorithm

**Standard SGD**:
1. Compute gradient: $g = \frac{1}{B} \sum_{i=1}^B \nabla \ell(x_i, \theta)$
2. Update: $\theta \leftarrow \theta - \eta g$

**DP-SGD** (Differentially Private SGD):
1. Compute per-example gradients: $g_i = \nabla \ell(x_i, \theta)$
2. **Clip** each gradient: $\bar{g}_i = g_i / \max(1, \|g_i\|_2 / C)$
3. **Sum** clipped gradients: $\bar{g} = \sum_{i=1}^B \bar{g}_i$
4. **Add noise**: $\tilde{g} = \bar{g} + \mathcal{N}(0, \sigma^2 C^2 I)$
5. **Update**: $\theta \leftarrow \theta - \frac{\eta}{B} \tilde{g}$

Where:
- $C$ = clipping norm (sensitivity bound)
- $\sigma$ = noise multiplier (typically 0.5-2.0)
- $B$ = batch size
- $\eta$ = learning rate

**Important**:
- Noise stddev = `noise_multiplier × clip_norm`
- Noise is added to the **sum**, not individual gradients
- We still normalize by batch size (for learning rate stability)

---

## Part 3: Adding Noise to Clipped Gradients

Let's extend our gradient clipping example from Tutorial 01 with noise.

### Alternative: Bounded Gaussian Noise

If you need noisy values to stay within a specific range (e.g., valid gradient bounds), use
`bounded_gaussian_noise()`. It samples from a **truncated normal** distribution restricted to `[lower, upper]`
([Chen & Hale, 2024](https://arxiv.org/abs/2211.17230)).

In [ ]:
# Bounded Gaussian: outputs guaranteed within [-3, 3]
bounded_fn, state = bounded_gaussian_noise(stddev=1.0, bounds=(-3.0, 3.0))
noisy_bounded, state = bounded_fn(grad, state)

print(f"Original gradient:     {grad}")
print(f"Bounded noisy gradient: {noisy_bounded}")
print(f"All values in [-3, 3]: {(noisy_bounded >= -3.0).all() and (noisy_bounded <= 3.0).all()}")

# Compare: standard gaussian has no bounds
unbounded_fn, state2 = gaussian_noise(stddev=1.0)
noisy_unbounded, state2 = unbounded_fn(grad, state2)
print(f"\nUnbounded noisy gradient: {noisy_unbounded}")
print(f"(may exceed [-3, 3])")

In [6]:
# Setup: Model and data (same as Tutorial 01)
class SimpleMLP(nn.Module):
  def __init__(self, input_dim=10):
    super().__init__()
    self.fc1 = nn.Linear(input_dim, 64)
    self.fc2 = nn.Linear(64, 32)
    self.fc3 = nn.Linear(32, 1)

  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    return self.fc3(x).squeeze(-1)


# Generate data
def generate_data(n_samples=10000, input_dim=10, seed=42):
  torch.manual_seed(seed)
  X = torch.randn(n_samples, input_dim)
  y = (X[:, 0] > 0).float()
  return X, y


X_train, y_train = generate_data()
print(f"Data: {X_train.shape}")

Data: torch.Size([200, 10])


In [7]:
# Convert to functional
model = SimpleMLP()
fmodel, params = make_functional(model)


# Define loss
def loss_fn(params, example):
  x, y = example
  logit = fmodel(params, x)
  return F.binary_cross_entropy_with_logits(logit, y)


# Create clipped gradient function
clip_norm = 1.0
clipped_grad_fn = clipped_grad(
  loss_fn,
  argnums=0,
  batch_argnums=1,
  l2_clip_norm=clip_norm,
)

print(f"Clipping norm: {clip_norm}")
print(f"Sensitivity bound: {clipped_grad_fn.l2_norm_bound}")

Clipping norm: 1.0
Sensitivity bound: 1.0


In [29]:
# Compute clipped gradients for a batch
X_batch = X_train[:32]
y_batch = y_train[:32]

clipped_grads = clipped_grad_fn(params, (X_batch, y_batch))

print("Clipped gradients (summed over batch):")
for i, g in enumerate(clipped_grads):
  print(f"  param[{i}]: shape {g.shape}, L2 norm {g.norm().item():.4f}")

Clipped gradients (summed over batch):
  param[0]: shape torch.Size([64, 10]), L2 norm 1.6634
  param[1]: shape torch.Size([64]), L2 norm 0.6243
  param[2]: shape torch.Size([32, 64]), L2 norm 5.4939
  param[3]: shape torch.Size([32]), L2 norm 2.0269
  param[4]: shape torch.Size([1, 32]), L2 norm 5.3438
  param[5]: shape torch.Size([1]), L2 norm 5.6306


### Add Gaussian Noise

Now let's add noise calibrated for differential privacy:

In [ ]:
# Noise parameters
noise_multiplier = 1.0
stddev = noise_multiplier * clip_norm  # stddev = noise_multiplier × sensitivity

print(f"Noise multiplier: {noise_multiplier}")
print(f"Clip norm (sensitivity): {clip_norm}")
print(f"Noise stddev: {stddev}")

# Add noise to clipped gradients
noise_fn, noise_state = gaussian_noise(stddev=stddev, generator=42)
noisy_grads, noise_state = noise_fn(clipped_grads, noise_state)

print("\nNoisy gradients:")
for i, (clean_g, noisy_g) in enumerate(zip(clipped_grads, noisy_grads)):
  noise_norm = (noisy_g - clean_g).norm().item()
  print(f"  param[{i}]: clean norm {clean_g.norm().item():.4f}, "
        f"noisy norm {noisy_g.norm().item():.4f}, "
        f"noise norm {noise_norm:.4f}")

### Understanding the Privacy-Utility Tradeoff

Let's see how different noise multipliers affect gradient quality:

In [ ]:
# Experiment: Add different amounts of noise to the same clipped gradient
clip_norm = 1.0
noise_multipliers = [0.5, 1.0, 2.0, 5.0]

# Use the clipped gradients from before
print(f"Original clipped gradient norm: {clipped_grads[0].norm().item():.4f}\n")

for nm in noise_multipliers:
  stddev = nm * clip_norm
  noise_fn, state = gaussian_noise(stddev=stddev, generator=42)
  noisy_grad, state = noise_fn(clipped_grads[0], state)

  noise_added = (noisy_grad - clipped_grads[0]).norm().item()
  signal_to_noise = clipped_grads[0].norm().item() / noise_added

  print(f"Noise multiplier = {nm:.1f}:")
  print(f"  Noise stddev: {stddev:.2f}")
  print(f"  Noise added (L2): {noise_added:.2f}")
  print(f"  Signal-to-noise ratio: {signal_to_noise:.2f}")
  print(f"  → {'Strong signal' if signal_to_noise > 1 else 'Weak signal (noise dominates)'}\n")

**Key observations**:
- Higher noise multiplier = more noise = worse gradient signal
- When signal-to-noise < 1, noise dominates (training will be difficult)
- When signal-to-noise > 1, signal is stronger (better training, less privacy)
- This is the **privacy-utility tradeoff**!

**Question**: How do we choose the right noise multiplier?
**Answer**: Privacy accounting + calibration (next sections)!

---

**Observations**:
- Noise magnitude is comparable to gradient magnitude
- This is necessary for privacy, but reduces gradient quality
- Higher noise_multiplier = better privacy, worse utility

---

## Part 4: Privacy Accounting

**Key question**: How much privacy budget (ε, δ) did we spend?

Privacy accounting tracks the cumulative privacy cost over multiple training steps.

### Functional Privacy Accounting API

Opaque uses a **functional** (immutable) approach to privacy accounting:

```python
# Create initial state (zero privacy cost)
state = acc.create()

# Compose mechanisms (returns NEW state)
state = acc.compose_poisson_gaussian(
    state,
    noise_multiplier=1.0,
    sample_rate=0.01,
    count=100,
)

# Query privacy (doesn't mutate state)
epsilon = acc.get_epsilon(state, delta=1e-5)
```

**Benefits**:
- Immutable state = no hidden mutations
- Can cache, checkpoint, and branch states
- Pure functions = easier to reason about
- Composable = can combine multiple mechanisms

### Different Sampling Methods

1. **Poisson sampling** (`compose_poisson_gaussian`):
   - Random batch sizes following Poisson distribution
   - Standard for DP-SGD theory
   - Most common in literature

2. **Fixed-size sampling** (`compose_sampled_gaussian`):
   - Fixed batch size (deterministic)
   - Slightly different privacy analysis
   - Simpler to implement

3. **Truncated Poisson** (`compose_truncated_poisson_gaussian`):
   - Bounded batch size (best of both worlds!)
   - Tighter privacy bounds than worst-case
   - Practical for real systems

Let's see each in action!

In [31]:
# Create privacy state (initially zero cost)
state = acc.create()

# Check initial privacy
epsilon_initial = acc.get_epsilon(state, delta=1e-5)
print(f"Initial state:")
print(f"  Epsilon (ε): {epsilon_initial:.4f}")
print(f"  Delta (δ): 1e-5")
print(f"  (This is the identity - no privacy spent yet)")

RDP Accountant created!
Initial epsilon: 0.0000


In [11]:
# Compose a single Poisson-sampled Gaussian mechanism
noise_multiplier = 1.1
sample_rate = 32 / 10000  # batch_size / dataset_size

# Compose returns a NEW state (functional!)
state_1step = acc.compose_poisson_gaussian(
  state,
  noise_multiplier=noise_multiplier,
  sample_rate=sample_rate,
  count=1,
)

epsilon_1 = acc.get_epsilon(state_1step, delta=1e-5)
print(f"After 1 step:")
print(f"  Epsilon (ε): {epsilon_1:.4f}")
print(f"  Delta (δ): 1e-5")
print(f"  Interpretation: (ε={epsilon_1:.2f}, δ=1e-5)-DP")

After 1 step(s):
  Epsilon (ε): 6.6463
  Delta (δ): 1e-5
  Interpretation: (ε=6.65, δ=1e-5)-DP


In [32]:
# Compose 99 more steps (total = 100)
state_100steps = acc.compose_poisson_gaussian(
  state_1step,
  noise_multiplier=noise_multiplier,
  sample_rate=sample_rate,
  count=99,
)

epsilon_100 = acc.get_epsilon(state_100steps, delta=1e-5)
print(f"After 100 steps:")
print(f"  Epsilon (ε): {epsilon_100:.4f}")
print(f"  Delta (δ): 1e-5")
print(f"\n  Privacy degraded from {epsilon_1:.2f} to {epsilon_100:.2f} (as expected!)")

# Original state is unchanged (immutability!)
print(f"\n  Original state still has ε = {acc.get_epsilon(state, delta=1e-5):.4f}")

After 100 steps:
  Epsilon (ε): 0.5638
  Delta (δ): 0.5e-4

  Privacy degraded from 6.65 to 0.56 (as expected!)


**Key insight**: Privacy budget **increases** (gets worse) with more training steps!

### Comparing Sampling Methods

Let's compare standard Poisson vs fixed-size vs truncated Poisson:

In [33]:
# Same parameters for fair comparison
noise_mult = 1.1
sample_rate = 32 / 10000
num_steps = 100

# Method 1: Poisson sampling (random batch sizes)
state_poisson = acc.create()
state_poisson = acc.compose_poisson_gaussian(
  state_poisson,
  noise_multiplier=noise_mult,
  sample_rate=sample_rate,
  count=num_steps,
)

# Method 2: Fixed-size sampling (deterministic batch size)
state_fixed = acc.create()
state_fixed = acc.compose_sampled_gaussian(
  state_fixed,
  noise_multiplier=noise_mult,
  batch_size=32,
  dataset_size=10000,
  count=num_steps,
)

# Method 3: Truncated Poisson (bounded random batch sizes)
state_truncated = acc.create()
state_truncated = acc.compose_truncated_poisson_gaussian(
  state_truncated,
  noise_multiplier=noise_mult,
  sample_rate=sample_rate,
  truncated_batch_size=32,
  dataset_size=10000,
  count=num_steps,
)

# Compare epsilons
eps_poisson = acc.get_epsilon(state_poisson, delta=1e-5)
eps_fixed = acc.get_epsilon(state_fixed, delta=1e-5)
eps_truncated = acc.get_epsilon(state_truncated, delta=1e-5)

print(f"After {num_steps} steps with noise_multiplier={noise_mult}:")
print(f"  Poisson:    ε = {eps_poisson:.4f}")
print(f"  Fixed-size: ε = {eps_fixed:.4f}")
print(f"  Truncated:  ε = {eps_truncated:.4f}")
print(f"\n  Truncated Poisson has best bounds (lowest ε for same utility)!")

After 100 steps:
  RDP: ε = 0.5638
  PLD: ε = 7.0284

  PLD is tighter by -6.4646 epsilon!
  (Lower epsilon = better privacy for same utility)


### Why Truncated Poisson is Better

Standard Poisson sampling has **variable batch sizes**:
- Can be 0 (rare but possible)
- Can be 2× or 3× the expected size
- Unpredictable memory usage
- Conservative worst-case privacy analysis

**Truncated Poisson** solves this:
- Batch size bounded to `truncated_batch_size`
- Still maintains randomness (DP requirement)
- Tighter privacy bounds = better utility
- Predictable resource usage

**Use truncated Poisson when**:
- You need stable batch sizes
- You want best privacy-utility tradeoff
- You're implementing production DP-SGD

In [ ]:
# Visualize privacy degradation over steps
steps_range = [1, 10, 50, 100, 200, 500]
epsilons_poisson = []
epsilons_truncated = []

for n_steps in steps_range:
  # Poisson
  s_p = acc.create()
  s_p = acc.compose_poisson_gaussian(s_p, noise_multiplier=1.1, sample_rate=0.0032, count=n_steps)
  epsilons_poisson.append(acc.get_epsilon(s_p, delta=1e-5))

  # Truncated
  s_t = acc.create()
  s_t = acc.compose_truncated_poisson_gaussian(
    s_t, noise_multiplier=1.1, sample_rate=0.0032,
    truncated_batch_size=32, dataset_size=10000, count=n_steps
  )
  epsilons_truncated.append(acc.get_epsilon(s_t, delta=1e-5))

plt.figure(figsize=(10, 6))
plt.plot(steps_range, epsilons_poisson, 'o-', label='Poisson', linewidth=2, markersize=8)
plt.plot(steps_range, epsilons_truncated, 's-', label='Truncated Poisson', linewidth=2, markersize=8)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Epsilon (ε)', fontsize=12)
plt.title('Privacy Degradation: Poisson vs Truncated Poisson', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Truncated Poisson consistently gives tighter bounds!")

---

## Part 5: Calibrating Noise

**Problem**: How do we choose `noise_multiplier` to achieve a target privacy budget?

**Solution**: Use calibration functions!

### Calibrate Noise Multiplier

Given:
- Target (ε, δ)
- Training parameters (sample_rate, num_steps)

Find:
- Minimum `noise_multiplier` that achieves target privacy

In [35]:
# Calibrate noise for target privacy
target_epsilon = 3.0
target_delta = 1e-5
sample_rate = 32 / 10000
num_steps = 1000

# Find noise multiplier for target (ε, δ)
noise_mult = acc.find_noise_multiplier_for_epsilon_delta(
  epsilon=target_epsilon,
  delta=target_delta,
  sample_rate=sample_rate,
  num_steps=num_steps,
)

print(f"Target privacy: (ε={target_epsilon}, δ={target_delta})")
print(f"Training: sample_rate={sample_rate:.4f}, num_steps={num_steps}")
print(f"\nCalibrated noise_multiplier: {noise_mult:.4f}")

Target privacy: (ε=3.0, δ=1e-05)
Training: sample_rate=0.0032, num_steps=1000

Calibrated noise_multiplier: 0.9845


In [ ]:

Verification:
Target
ε: 3.0000
Achieved
ε: 2.9987
Difference: 0.0013

✓ Calibration
successful!


### Unified Calibration API: sampling_method Parameter

**New in this release!** All calibration functions now support a unified `sampling_method` parameter:
- `"poisson"` - Standard Poisson sampling (default)
- `"fixed_batch"` - Fixed-size batch sampling
- `"truncated_poisson"` - Truncated Poisson sampling (recommended!)

This allows you to easily compare and choose the best sampling method for your use case.

In [ ]:
# Compare calibration across different sampling methods
target_eps = 3.0
target_delta = 1e-5
num_steps = 1000

# Method 1: Poisson (default)
noise_poisson = acc.find_noise_multiplier_for_epsilon_delta(
  epsilon=target_eps,
  delta=target_delta,
  sampling_method="poisson",
  sample_rate=32 / 10000,
  num_steps=num_steps,
)

# Method 2: Fixed batch
noise_fixed = acc.find_noise_multiplier_for_epsilon_delta(
  epsilon=target_eps,
  delta=target_delta,
  sampling_method="fixed_batch",
  batch_size=32,
  dataset_size=10000,
  num_steps=num_steps,
)

# Method 3: Truncated Poisson (recommended!)
noise_truncated = acc.find_noise_multiplier_for_epsilon_delta(
  epsilon=target_eps,
  delta=target_delta,
  sampling_method="truncated_poisson",
  sample_rate=32 / 10000,
  truncated_batch_size=32,
  dataset_size=10000,
  num_steps=num_steps,
)

print(f"Calibrated noise multipliers for ε={target_eps}, δ={target_delta}:")
print(f"  Poisson:           {noise_poisson:.4f}")
print(f"  Fixed-batch:       {noise_fixed:.4f}")
print(f"  Truncated Poisson: {noise_truncated:.4f}")
print(f"\nAll achieve same privacy! Choose based on your batch size requirements.")

In [ ]:
# Verify that all methods achieve the target privacy
print(f"\nVerification (all should achieve ε ≈ {target_eps}):")

# Verify Poisson
state_p = acc.create()
state_p = acc.compose_poisson_gaussian(
  state_p, noise_multiplier=noise_poisson,
  sample_rate=32 / 10000, count=num_steps
)
eps_p = acc.get_epsilon(state_p, delta=target_delta)
print(f"  Poisson:           ε = {eps_p:.4f} (error: {abs(eps_p - target_eps):.4f})")

# Verify Fixed-batch
state_f = acc.create()
state_f = acc.compose_sampled_gaussian(
  state_f, noise_multiplier=noise_fixed,
  batch_size=32, dataset_size=10000, count=num_steps
)
eps_f = acc.get_epsilon(state_f, delta=target_delta)
print(f"  Fixed-batch:       ε = {eps_f:.4f} (error: {abs(eps_f - target_eps):.4f})")

# Verify Truncated Poisson
state_t = acc.create()
state_t = acc.compose_truncated_poisson_gaussian(
  state_t, noise_multiplier=noise_truncated,
  sample_rate=32 / 10000, truncated_batch_size=32,
  dataset_size=10000, count=num_steps
)
eps_t = acc.get_epsilon(state_t, delta=target_delta)
print(f"  Truncated Poisson: ε = {eps_t:.4f} (error: {abs(eps_t - target_eps):.4f})")

print(f"\n✓ All methods successfully calibrated to target privacy!")

**Key insight**: The unified `sampling_method` API makes it easy to switch between different sampling strategies without changing your code structure. All three methods achieve the same target privacy, but have different implementation tradeoffs:

- **Poisson**: Standard in DP theory, simple to implement
- **Fixed-batch**: Deterministic batch sizes, simpler for some systems
- **Truncated Poisson**: Best of both worlds - tighter bounds with stable batches

In practice, **truncated Poisson is recommended** for production DP-SGD as it provides better privacy-utility tradeoffs with predictable resource usage.

### Other Calibration Functions

Opaque provides three calibration functions for different privacy metrics:

1. **`find_noise_multiplier_for_epsilon_delta()`** - Standard (ε, δ)-DP (what we just used)
2. **`find_noise_multiplier_for_advantage()`** - f-DP advantage metric
3. **`find_noise_multiplier_for_err_rates()`** - (α, β) error rates

All use the same pattern: given target privacy and training params, find optimal noise.

### Example: Calibrating for Error Rates

The (α, β) framework models privacy as hypothesis testing:
- **α (alpha)**: False positive rate (Type I error)
- **β (beta)**: False negative rate (Type II error)

Given α, find noise such that β ≤ target_beta.

In [37]:
# Calibrate for error rates (α, β)
alpha = 1e-4  # False positive rate
target_beta = 0.8  # Target false negative rate
sample_rate = 16 / 2048
num_steps = 1000

# Find noise multiplier
noise_mult_beta = acc.find_noise_multiplier_for_err_rates(
  alpha=alpha,
  beta=target_beta,
  sample_rate=sample_rate,
  num_steps=num_steps,
)

print(f"Target: β ≤ {target_beta} at α = {alpha}")
print(f"Training: sample_rate={sample_rate:.4f}, num_steps={num_steps}")
print(f"\nCalibrated noise_multiplier: {noise_mult_beta:.4f}")

# Verify
achieved_beta = acc.get_beta_for_dpsgd(
  noise_multiplier=noise_mult_beta,
  sample_rate=sample_rate,
  num_steps=num_steps,
  alpha=alpha,
)

print(f"\nVerification:")
print(f"  Target β: {target_beta:.4f}")
print(f"  Achieved β: {achieved_beta:.4f}")
print(f"  ✓ Success!" if achieved_beta <= target_beta + 0.01 else "  ✗ Failed")

Target: β ≤ 0.8 at α = 0.0001
Training: sample_rate=0.0078, num_steps=1000

Calibrated noise_multiplier: 0.6234

Verification:
  Target β: 0.8000
  Achieved β: 0.7996
  ✓ Success!


---

## Summary

### What We Learned

1. **Differential Privacy Basics**
   - (epsilon, delta)-DP provides formal privacy guarantees
   - Requires both clipping (sensitivity) and noise (randomness)
   - Lower epsilon = better privacy, but harder to achieve

2. **Noise Injection with Immutable State**
   - `gaussian_noise(stddev, generator=...)` returns `(noise_fn, state)`
   - `noise_fn(grads, state) -> (noisy_grads, new_state)`
   - `bounded_gaussian_noise(stddev, bounds, generator=...)` - Bounded noise via truncated normal ([Chen & Hale, 2024](https://arxiv.org/abs/2211.17230))
   - Noise stddev = `noise_multiplier * clip_norm`
   - Noise is added to **summed clipped gradients**, not per-example
   - Unbiased: noise is centered around true value
   - Generator options: `None` (unseeded), `int` (seeded), `torch.Generator`

3. **Functional Privacy Accounting**
   - **Immutable state**: `state = acc.create()`
   - **Composition**: `state = acc.compose_poisson_gaussian(state, ...)`
   - **Query**: `epsilon = acc.get_epsilon(state, delta=...)`
   - Privacy budget (epsilon) **increases** (degrades) with more training steps
   - Three sampling methods:
     - `compose_poisson_gaussian()` - Standard Poisson sampling
     - `compose_sampled_gaussian()` - Fixed-size batches
     - `compose_truncated_poisson_gaussian()` - Bounded Poisson (best!)

4. **Truncated Poisson Sampling**
   - Bounds batch size while maintaining DP guarantees
   - Solves variable batch size problem of standard Poisson sampling
   - Provides tighter privacy bounds = better utility
   - Use when you need stable, predictable batch sizes

5. **Calibration Functions** (using riskcal)
   - **`find_noise_multiplier_for_epsilon_delta()`**: Standard (epsilon, delta)-DP
   - **`find_noise_multiplier_for_advantage()`**: f-DP advantage metric
   - **`find_noise_multiplier_for_err_rates()`**: (alpha, beta) hypothesis testing
   - Automated hyperparameter tuning for target privacy
   - All follow same pattern: target -> optimal noise

### Privacy Metrics Comparison

| Metric | Description | Use Case |
|--------|-------------|----------|
| **(epsilon, delta)-DP** | Traditional differential privacy | Standard privacy reporting |
| **Advantage** | f-DP attack advantage | Modern interpretable metric |
| **(alpha, beta) Error Rates** | Hypothesis testing framework | Operational risk assessment |

All three are equivalent - just different ways to express privacy!

### Complete DP-SGD Workflow

```python
import opaque.accounting as acc
from opaque import clipped_grad, gaussian_noise

# 1. Setup: Calibrate noise for target privacy
noise_multiplier = acc.find_noise_multiplier_for_epsilon_delta(
    epsilon=3.0,
    delta=1e-5,
    sample_rate=batch_size / dataset_size,
    num_steps=total_steps,
)

# 2. Create noise function (configure once)
privacy_state = acc.create()
clip_norm = 1.0
noise_fn, noise_state = gaussian_noise(
    stddev=noise_multiplier * clip_norm, generator=42
)

# 3. Training loop
for step in range(total_steps):
    # Compute clipped gradients
    clipped_grads = clipped_grad_fn(params, batch)

    # Add calibrated noise
    noisy_grads, noise_state = noise_fn(clipped_grads, noise_state)

    # Update parameters
    params = optimizer_step(params, noisy_grads)

    # Track privacy (compose returns new state)
    privacy_state = acc.compose_poisson_gaussian(
        privacy_state,
        noise_multiplier=noise_multiplier,
        sample_rate=batch_size / dataset_size,
        count=1,
    )

    # Check privacy spent
    if step % 100 == 0:
        epsilon = acc.get_epsilon(privacy_state, delta=1e-5)
        print(f"Step {step}: epsilon = {epsilon:.2f}")
```

---

## What's Next?

**Tutorial 03: Complete DP-SGD Training**
- Put it all together: full DP-SGD training loop
- Compare private vs non-private training
- Visualize privacy-utility tradeoff
- Train on real tasks

**Tutorial 04: DP LoRA Fine-Tuning**
- Apply DP-SGD to fine-tune LLMs with LoRA
- Real-world privacy-preserving ML
- Integration with HuggingFace

---

## Exercises

1. **Noise exploration**: Try different noise multipliers (0.5, 1.0, 2.0) and visualize their distributions using `gaussian_noise(generator=seed)`

2. **Sampling comparison**: Compare Poisson vs truncated Poisson privacy bounds for different sample rates

3. **Calibration practice**:
   - Find noise for epsilon=1.0, delta=1e-5, 1000 steps, sample_rate=0.001
   - Find noise for advantage=0.1 with same parameters
   - Find noise for beta=0.9 at alpha=0.01 with same parameters

4. **Privacy budget analysis**: Plot how epsilon changes with number of steps for different noise multipliers

5. **State branching**: Create a privacy state, compose 100 steps, then branch to explore two different futures

---

## Key Takeaways

- **DP = Clipping + Noise + Accounting** - all three are necessary!
- **Immutable state API** - `noise_fn, state = gaussian_noise(...)` then `noisy, state = noise_fn(grads, state)`
- **Functional accounting** - immutable state, pure functions, composable
- **Truncated Poisson is best** - stable batches + tight bounds
- **Calibration automates tuning** - no manual trial-and-error

---

Questions? Open an issue on [GitHub](https://github.com/evgri243/opaque/issues).

**Further Reading**:
- [Deep Learning with Differential Privacy](https://arxiv.org/abs/1607.00133) (Abadi et al., 2016)
- [The Bounded Gaussian Mechanism for Differential Privacy](https://arxiv.org/abs/2211.17230) (Chen & Hale, 2024)
- [Privacy Loss Distribution](https://arxiv.org/abs/2106.08567) (Koskela et al., 2021)
- [Truncated Poisson Sampling](https://arxiv.org/abs/2508.15089) (Zanella-Béguelin et al., 2025)